<a href="https://colab.research.google.com/github/juanitapinedaguti/Integracion-de-Datos-y-Prospectiva/blob/main/Integracion_Multidimensional_parte_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Caso de estudio-integracion vertical EPS**

####Ya que se van a cerrar las sedes de Caldas y Copacabana, vamos a revisar a donde vamos a mandar a lois clientes de estas sedes tenineod en cuenta las siguientes variables:

*   **Pregnancies**: Número de embarazos.
*   **Glucose**: Concentración de glucosa en plasma 2 horas después de una prueba de tolerancia oral a la glucosa.
*   **BloodPressure**: Presión arterial diastólica (mm Hg).
*   **SkinThickness**: Grosor del pliegue cutáneo del tríceps (mm).
*   **Insulin**: Insulina sérica de 2 horas (mu U/ml).
*   **BMI**: Índice de masa corporal (peso en kg/(altura en m)^2).
*   **DiabetesPedigreeFunction**: Función de pedigrí de diabetes.
*   **Age**: Edad en años.
*   **Numero_Atenciones**: Número de atenciones/citas médicas.
*   **Costo_Promedio_Atencion**: Costo promedio de atención.
*   **Ciudad_Pertenencia**: Ciudad de pertenencia (variable categórica que indica la ciudad del paciente).
*   **Outcome**: Un resultado binario que indica la presencia o ausencia de diabetes (1 o 0).


# 0. Cargar la librerías de trabajo

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Carga de base de datos

In [28]:
nxl="/content/drive/MyDrive/INTEGRACION Y PROSPECTIVA/5. Diabetes Árbol_Int_Mult.xlsx"
XDB=pd.read_excel(nxl,sheet_name=0)
XDB.dropna()
XDB.head()

XDB2=XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10,8,11]].copy() # Acá están TODAS las variables
XDB2.head()

XDB=XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10]].copy() # Acá solo están las variables de entrada
XDB.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion
0,0,125,96,0,0,22.5,0.262,21,4,314.926044
1,0,141,0,0,0,42.4,0.205,29,6,287.355812
2,10,101,86,37,0,45.6,1.136,38,8,464.541158
3,1,96,122,0,0,22.4,0.207,27,3,368.103292
4,5,139,64,35,140,28.6,0.411,26,9,428.548590


# 2. Creación de los clusters para las EPS de los municipios que van a quedar activos

In [63]:
Xciu=XDB2['Ciudad_Pertenencia'].unique()
Xciu=['Sabaneta', 'Envigado', 'Medellín', 'Bello'] # Estos serán los concentradores de información

# Creación de clusters
XCm=np.zeros((len(Xciu),10)) # No se incluyen las variable Outcome y Ciudad_Pertenencia
XDB_features=XDB.copy()

#Fase 0 de integración: reconocer los valores de entrada por ciudad
for i, city_name in enumerate(Xciu):
  print(i,city_name)

  filas=np.where(XDB2['Ciudad_Pertenencia']==city_name)[0]
  # filas # This line was just for debugging, it can be removed or commented out

  XCm[i,:]=np.mean(XDB.iloc[filas,:],axis=0)

XCma=XCm.copy()
display(XCma)


0 Sabaneta
1 Envigado
2 Medellín
3 Bello


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion
0,3.872685,122.824074,67.907407,19.967593,77.511574,31.730093,0.491333,33.488426,5.557870,273.684041
1,3.805699,121.207254,68.829016,20.730570,79.409326,32.046114,0.460345,32.525907,5.515544,282.643785
2,3.820862,120.746032,70.287982,20.977324,82.947846,31.569388,0.489918,33.299320,5.421769,273.701485
3,4.123874,122.542793,70.085586,20.614865,78.459459,32.285135,0.481115,33.774775,5.574324,280.357432


# 3. Integración de los pacientes de Caldas y Copacabana

In [79]:
Xciu2=['Caldas', 'Copacabana'] # Estos serán los pacientes
Xciu3=[] # Aquí se va a almacenar en que EPS quedaron los pacientes

for k in range (len(XDB2)): # Contiene todas las variables
  # print(XDB2.iloc[k,11]) # A donde pertenece cada paciente

  if XDB2.iloc[k,11]=='Caldas' or XDB2.iloc[k,11]=='Copacabana': # Corregido para comparar con strings directamente
    d=np.sum( (XCm-XDB.iloc[k, :].values)**2, axis=1) #distancia de cada paciente a cada cluster
    nc=np.argmin(d)

    # print(Xciu[nc])
    Xciu3.append(Xciu[nc])
    XCm[nc,]=(XCm[nc,:]+XDB.iloc[k, :].values)/2
  else:
    Xciu3.append('--') # Mantener la ciudad original si no es Caldas o Copacabana

XDB2['Transferencia']=Xciu3 # Asignar la lista completa al final del bucle
#display(XDB2)

XCmd=XCm.copy()

diferencia=pd.DataFrame(((XCmd-XCma)/XCmd),columns=XDB.columns) # Diferencia antes y despues
display(diferencia)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion
0,-3.279529,0.040835,-0.006269,0.429604,0.388358,0.140545,-0.687157,-0.379931,-0.192042,0.268690
1,-0.182947,0.074212,0.130619,0.494136,0.791606,0.135837,0.250403,-0.064088,0.394290,-1.474541
2,0.394699,0.132078,-0.275610,-1.328412,-14.397988,0.117276,0.160250,-0.121329,-0.498931,0.262479
3,-0.123090,-0.060963,0.066046,-0.250262,0.293048,-0.111180,-0.188445,-0.030012,0.129226,-0.932413


In [51]:
print("Distribución inicial de pacientes por ciudad:")
initial_counts = XDB2['Ciudad_Pertenencia'].value_counts().rename('Inicial')
print(initial_counts)

print("\nDistribución final de pacientes por ciudad (después de la transferencia):")
final_counts = XDB2['Transferencia'].value_counts().rename('Final')
print(final_counts)

# Para una comparación lado a lado
comparison_df = pd.concat([initial_counts, final_counts], axis=1).fillna(0)
comparison_df['Cambio'] = comparison_df['Final'] - comparison_df['Inicial']
print("\nComparativo de pacientes por ciudad:")
display(comparison_df.astype(int))

Distribución inicial de pacientes por ciudad:
Ciudad_Pertenencia
Bello         444
Medellín      441
Sabaneta      432
Caldas        414
Envigado      386
Copacabana    383
Name: Inicial, dtype: int64

Distribución final de pacientes por ciudad (después de la transferencia):
Transferencia
--          1703
Envigado     340
Sabaneta     253
Medellín     180
Bello         24
Name: Final, dtype: int64

Comparativo de pacientes por ciudad:


,Inicial,Final,Cambio
Bello,444,24,-420
Medellín,441,180,-261
Sabaneta,432,253,-179
Caldas,414,0,-414
Envigado,386,340,-46
Copacabana,383,0,-383
--,0,1703,1703
